# Step 1-4: Pull, filter, and export accredited institutions

Pulls institution-level data from the College Scorecard API (`api.data.gov/ed/collegescorecard`),
which only includes institutions eligible for Title IV federal student aid — i.e. accredited by a
USDE-recognized accreditor. We then keep only institutions whose primary institutional accreditor is
**also** recognized by CHEA (see `../reference/chea_usde_institutional_accreditors.csv`), drop
for-profit institutions, drop institutions reporting a religious affiliation, and export the
remaining institutions (name + homepage URL) to a spreadsheet.

Requires a free API key from https://api.data.gov/signup/, provided either way:

- **`COLLEGE_SCORECARD_API_KEY` environment variable** — checked first. This is the path for
  CI/automated runs: store the key as a GitHub Actions secret and expose it to the job as this
  environment variable (e.g. `env: COLLEGE_SCORECARD_API_KEY: ${{ secrets.COLLEGE_SCORECARD_API_KEY }}`
  in the workflow YAML).
- **`credentials.json`** at the repo root (`resource_listings/credentials.json`, gitignored — never
  commit it) — the fallback for local/interactive use, read only if the environment variable above
  isn't set:

```json
{
  "credentials": {
    "email": "you@example.com",
    "key": "your-api-key"
  }
}
```

The key's default tier is limited to 1,000 requests/hour (shared across the College Scorecard API
and every other api.data.gov API using the same key) — the fetch loop below paces requests well
under that and backs off automatically on 429 responses.

In [1]:
import json
import os
import pathlib
import time
import requests
import pandas as pd

# credentials.json lives at the repo root (resource_listings/credentials.json) and is gitignored.
CREDENTIALS_PATH = pathlib.Path("../../credentials.json")
API_KEY_ENV_VAR = "COLLEGE_SCORECARD_API_KEY"

def load_college_scorecard_key(path=CREDENTIALS_PATH, env_var=API_KEY_ENV_VAR):
    # Environment variable takes priority -- this is how a GitHub Actions secret reaches the
    # notebook in CI (mapped to this env var via the workflow's `env:` block). Falls back to
    # credentials.json for local/interactive use, where committing a secret file isn't an option
    # but a GH secret isn't set up either.
    env_key = os.environ.get(env_var)
    if env_key:
        return env_key

    text = path.read_text(encoding="utf-8").strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        # tolerate a bare `"credentials": {...}` block missing its enclosing braces
        data = json.loads("{" + text + "}")
    creds = data.get("credentials", data)
    key = creds.get("key")
    if not key:
        raise ValueError(f"No 'key' field found in {path}, and {env_var} is not set")
    return key

API_KEY = load_college_scorecard_key()

BASE_URL = "https://api.data.gov/ed/collegescorecard/v1/schools"
FIELDS = [
    "id",
    "school.name",
    "school.accreditor",
    "school.ownership",
    "school.religious_affiliation",
    "school.school_url",
    "school.city",
    "school.state",
    "school.degrees_awarded.predominant",
    # NOTE: program_percentage fields require the "latest." prefix -- the bare
    # "academics.program_percentage.*" alias silently returns null for every institution (verified
    # against the live API: even Juilliard, ~100% performing arts, returned null without this
    # prefix). "school.*" fields are top-level aliases and don't need it; "academics.*" fields are
    # nested under "latest" in the raw API response and do.
    "latest.academics.program_percentage.visual_performing",
    "latest.academics.program_percentage.theology_religious_vocation",
    "latest.academics.program_percentage.health",
    "latest.academics.program_percentage.biological",
]
PER_PAGE = 100

# api.data.gov's default key tier allows 1,000 requests/hour (returned as X-RateLimit-Limit on
# every response; exceeding it gets a 429). Pulling ~6,000 institutions at PER_PAGE=100 only takes
# ~60 requests, but we still pace requests conservatively below that ceiling and back off on 429s
# in case the dataset grows or the key's limit is ever tightened.
RATE_LIMIT_PER_HOUR = 1000
MIN_SECONDS_BETWEEN_REQUESTS = 3600 / RATE_LIMIT_PER_HOUR  # 3.6s
MAX_RETRIES = 5

In [2]:
def fetch_page(page):
    """GET one page, retrying with backoff on 429s (respecting Retry-After when present)."""
    for attempt in range(MAX_RETRIES):
        resp = requests.get(
            BASE_URL,
            params={
                "api_key": API_KEY,
                "fields": ",".join(FIELDS),
                "per_page": PER_PAGE,
                "page": page,
            },
            timeout=30,
        )
        if resp.status_code == 429:
            wait = float(resp.headers.get("Retry-After", 2 ** attempt))
            print(f"429 on page {page}, backing off {wait:.1f}s (attempt {attempt + 1}/{MAX_RETRIES})")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp
    raise RuntimeError(f"Gave up on page {page} after {MAX_RETRIES} consecutive 429s")


def fetch_all_institutions():
    records = []
    page = 0
    while True:
        resp = fetch_page(page)
        payload = resp.json()
        results = payload.get("results", [])
        if not results:
            break
        records.extend(results)
        total = payload.get("metadata", {}).get("total", 0)
        if len(records) >= total:
            break

        # If the API tells us we're nearly out of quota for this window, wait it out rather than
        # racing toward a 429; otherwise just hold to the conservative per-request pace.
        remaining = resp.headers.get("X-RateLimit-Remaining")
        if remaining is not None and int(remaining) <= 1:
            print("Rate limit nearly exhausted, pausing 60s before continuing")
            time.sleep(60)
        else:
            time.sleep(MIN_SECONDS_BETWEEN_REQUESTS)

        page += 1
    return records

raw_records = fetch_all_institutions()
len(raw_records)

6273

In [3]:
df = pd.json_normalize(raw_records)
df = df.rename(columns={
    "id": "unitid",
    "school.name": "institution_name",
    "school.accreditor": "accreditor_raw",
    "school.ownership": "ownership",
    "school.religious_affiliation": "religious_affiliation",
    "school.school_url": "homepage_url",
    "school.city": "city",
    "school.state": "state",
    "latest.academics.program_percentage.visual_performing": "pct_visual_performing_arts",
    "latest.academics.program_percentage.theology_religious_vocation": "pct_theology",
    "latest.academics.program_percentage.health": "pct_health",
    "latest.academics.program_percentage.biological": "pct_biological",
})
df.shape

(6273, 13)

## Filter 1: accredited by a CHEA + USDE dual-recognized accreditor

Every institution in this dataset is already USDE-accredited by definition (Title IV eligibility
requires it). We additionally require the accreditor name to match one on the dual-recognition
reference list. `accreditor_raw` sometimes contains more than one accreditor name (e.g. an
institutional accreditor plus a specialized/programmatic one) separated by a semicolon, so we do a
case-insensitive substring match rather than an exact-equality match.

In [4]:
accreditors_ref = pd.read_csv("../reference/chea_usde_institutional_accreditors.csv")
dual_names = [n.lower() for n in accreditors_ref["accreditor_name"]]

def matches_dual_accreditor(raw):
    if not isinstance(raw, str) or not raw.strip():
        return False
    raw_lower = raw.lower()
    return any(name in raw_lower for name in dual_names)

df["dual_accredited"] = df["accreditor_raw"].apply(matches_dual_accreditor)
df["dual_accredited"].value_counts()

dual_accredited
True     3392
False    2881
Name: count, dtype: int64

If this leaves unexpectedly few institutions, inspect `df[~df["dual_accredited"]]["accreditor_raw"]`
for spelling/abbreviation variants (e.g. "ACCJC", "WASC") not covered by the reference list and add
them to `../reference/chea_usde_institutional_accreditors.csv`.

## Filter 4: exclude dedicated art/music schools

Scope is biomedical science, so drop institutions that are predominantly art/music schools rather
than screen them out later by hand (this is how "Art Academy of Cincinnati" and "American Academy
of Dramatic Arts-New York" ended up in the 1000-institution test run). Two signals, either one
sufficient to exclude:

- `academics.program_percentage.visual_performing` (CIP 50, Visual and Performing Arts — covers
  fine art, design, film, theater, *and* music, so one field catches both) — majority of degrees
  awarded in this category.
- Institution name matches a small set of unambiguous arts/music-institution phrases, for the many
  small conservatories/art schools that don't report program-mix data at all (so the percentage
  field is null rather than low). Deliberately narrow multi-word phrases (not bare "art"/"arts")
  to avoid false-positives on comprehensive universities that merely have an arts department.

In [5]:
import re

ARTS_MUSIC_NAME_PATTERN = re.compile(
    r"art institute|art academy|academy of art|school of art\b|college of art\b|art & design|"
    r"art and design|art center|conservatory of music|school of music\b|college of music\b|"
    r"music college|dramatic arts|film school|fashion institute|"
    r"academy of dramatic|institute of art\b|school of design\b|college of design\b",
    re.IGNORECASE,
)

def is_arts_or_music_school(row):
    pct = row["pct_visual_performing_arts"]
    if pd.notna(pct) and pct >= 0.5:
        return True
    return bool(ARTS_MUSIC_NAME_PATTERN.search(row["institution_name"]))

df["is_arts_or_music_school"] = df.apply(is_arts_or_music_school, axis=1)
df["is_arts_or_music_school"].value_counts()

is_arts_or_music_school
False    6186
True       87
Name: count, dtype: int64

## Filter 5: tighten the religious-institution exclusion

`religious_affiliation` is self-reported to IPEDS and is often blank even for genuinely religious
schools — Asbury University (an evangelical Christian university) passed Filter 3 in the
1000-institution test run despite this. Two additional signals, either sufficient to exclude:

- `accreditor_raw` mentions a national faith-related accrediting organization (see
  `reference/national_faith_related_accreditors.csv` — sourced from CHEA's May 2021
  CHEA-USDE_Recognized_Organizations chart: ABHE, AARTS, AIJS, ATS, TRACS). These accreditors run
  *alongside* a school's regional accreditor, so a religious school can pass Filter 1 (dual
  ED+CHEA recognition) via its regional accreditor while also carrying one of these — a strong,
  independent religious signal Filter 1 doesn't catch.
- `academics.program_percentage.theology_religious_vocation` (CIP 39) >= 0.5 — catches seminaries
  and divinity schools directly, independent of both the IPEDS field and the accreditor list.

This does not catch every religious institution (e.g. Asbury has neither signal — no faith-related
accreditor and a low theology-program share, since it's a comprehensive university that happens to
be evangelical). A further, stricter pass (name-pattern matching plus a curated manual override
list, applied to this notebook's *output* rather than baked into this pipeline) is available as a
separate, independently-runnable skill — see
`../../institution-resource-list-cleanup/SKILL.md`.</cell id="9bf09ccf">


In [6]:
faith_accreditors_ref = pd.read_csv("../reference/national_faith_related_accreditors.csv")
faith_accreditor_names = [n.lower() for n in faith_accreditors_ref["accreditor_name"]]

def matches_faith_related_accreditor(raw):
    if not isinstance(raw, str) or not raw.strip():
        return False
    raw_lower = raw.lower()
    return any(name in raw_lower for name in faith_accreditor_names)

def is_religious_institution(row):
    if matches_faith_related_accreditor(row["accreditor_raw"]):
        return True
    pct = row["pct_theology"]
    return bool(pd.notna(pct) and pct >= 0.5)

df["is_religious_institution"] = df.apply(is_religious_institution, axis=1)
df["is_religious_institution"].value_counts()

is_religious_institution
False    6097
True      176
Name: count, dtype: int64

In [7]:
# Filter 2: non-profit only (ownership: 1=public, 2=private nonprofit, 3=private for-profit)
# Filter 3: non-religious only (no reported religious affiliation)
# Filter 4: not a dedicated art/music school
# Filter 5: not caught by the tightened religious-institution check
has_homepage = df["homepage_url"].notna() & (df["homepage_url"].str.strip() != "")

funnel = {
    "starting pool": len(df),
    "+ dual accredited (Filter 1)": int((df["dual_accredited"]).sum()),
    "+ non-profit (Filter 2)": int((df["dual_accredited"] & df["ownership"].isin([1, 2])).sum()),
    "+ non-religious, IPEDS field (Filter 3)": int((
        df["dual_accredited"] & df["ownership"].isin([1, 2]) & df["religious_affiliation"].isna()
    ).sum()),
    "+ not art/music school (Filter 4)": int((
        df["dual_accredited"] & df["ownership"].isin([1, 2]) & df["religious_affiliation"].isna()
        & ~df["is_arts_or_music_school"]
    ).sum()),
    "+ not religious, tightened check (Filter 5)": int((
        df["dual_accredited"] & df["ownership"].isin([1, 2]) & df["religious_affiliation"].isna()
        & ~df["is_arts_or_music_school"] & ~df["is_religious_institution"]
    ).sum()),
    "+ has homepage URL": int((
        df["dual_accredited"] & df["ownership"].isin([1, 2]) & df["religious_affiliation"].isna()
        & ~df["is_arts_or_music_school"] & ~df["is_religious_institution"] & has_homepage
    ).sum()),
}
for label, count in funnel.items():
    print(f"{count:5d}  {label}")

filtered = df[
    df["dual_accredited"]
    & df["ownership"].isin([1, 2])
    & df["religious_affiliation"].isna()
    & ~df["is_arts_or_music_school"]
    & ~df["is_religious_institution"]
    & has_homepage
].copy()

filtered = filtered.drop_duplicates(subset="unitid").sort_values("institution_name")
len(filtered)

 6273  starting pool
 3392  + dual accredited (Filter 1)
 3104  + non-profit (Filter 2)
 2444  + non-religious, IPEDS field (Filter 3)
 2401  + not art/music school (Filter 4)
 2400  + not religious, tightened check (Filter 5)
 2399  + has homepage URL


2399

## Prioritize (not filter) by biomedical relevance

Not a filter — every institution that passes Filters 1-5 stays in the list regardless of this
value. Health Professions (CIP 51) + Biological/Biomedical Sciences (CIP 26) program share is used
only to **sort** the export, so Step 5 (the slow, agentic, per-domain search) tackles the
highest-relevance institutions first and can stop early on the long tail if time runs out.

This has to stay a sort, not a filter: `pct_health`/`pct_biological` are computed from
**undergraduate** completions, so graduate-only professional schools have both fields null —
verified live against Albert Einstein College of Medicine and A T Still University of Health
Sciences (both null despite being medical schools), vs. Adelphi University (27.8% health, 10.9%
biological — has undergrad nursing/health programs, so it reports). A hard threshold filter would
have wrongly dropped exactly the standalone medical/health-professional schools most worth keeping.
Sorting sidesteps this: those institutions just won't sort to the top on this metric, but they're
never removed, and `has_biomedical_program_data` marks the null cases so it's visible in the
spreadsheet rather than silently reading as "0% relevant."

In [8]:
filtered["has_biomedical_program_data"] = filtered["pct_health"].notna() | filtered["pct_biological"].notna()
filtered["pct_biomedical"] = filtered["pct_health"].fillna(0) + filtered["pct_biological"].fillna(0)

print(f"{len(filtered)} institutions total")
print(f"{int(filtered['has_biomedical_program_data'].sum())} report undergraduate program-mix data "
      f"(health and/or biological science share available)")
print(f"{int((~filtered['has_biomedical_program_data']).sum())} do not (often graduate-only/professional "
      f"schools — not excluded, just not ranked highly by this metric)")
print()
print("Top 15 by pct_biomedical (health + biological program share):")
filtered.sort_values("pct_biomedical", ascending=False)[
    ["institution_name", "pct_health", "pct_biological", "pct_biomedical", "has_biomedical_program_data"]
].head(15)

2399 institutions total
2062 report undergraduate program-mix data (health and/or biological science share available)
337 do not (often graduate-only/professional schools — not excluded, just not ranked highly by this metric)

Top 15 by pct_biomedical (health + biological program share):


,institution_name,pct_health,pct_biological,pct_biomedical,has_biomedical_program_data
2734,University of Pittsburgh-Titusville,1.0000,0.0000,1.0,True
2837,Medical University of South Carolina,1.0000,0.0000,1.0,True
5178,The University of Tennessee Health Science Center,1.0000,0.0000,1.0,True
3131,The University of Texas Health Science Center ...,1.0000,0.0000,1.0,True
3109,The University of Texas Health Science Center ...,1.0000,0.0000,1.0,True
3873,The University of Texas MD Anderson Cancer Center,1.0000,0.0000,1.0,True
323,Los Angeles County College of Nursing and Alli...,1.0000,0.0000,1.0,True
2145,Upstate Medical University,1.0000,0.0000,1.0,True
1767,University of Nebraska Medical Center,1.0000,0.0000,1.0,True
3110,The University of Texas Medical Branch at Galv...,1.0000,0.0000,1.0,True


In [9]:
import urllib.parse
from datetime import date

def normalize_url(u):
    u = u.strip()
    if not u:
        return u
    if not u.lower().startswith(("http://", "https://")):
        u = "https://" + u
    return u

def base_domain(u):
    """Registrable-ish domain for grouping (strips scheme, 'www.', path, port)."""
    netloc = urllib.parse.urlparse(u).netloc.lower().split(":")[0]
    if netloc.startswith("www."):
        netloc = netloc[4:]
    return netloc

output = filtered[[
    "unitid", "institution_name", "homepage_url", "city", "state", "accreditor_raw",
    "pct_biomedical", "has_biomedical_program_data",
]].copy()
output["homepage_url"] = output["homepage_url"].apply(normalize_url)
output = output.rename(columns={"accreditor_raw": "accreditor"})
output["base_domain"] = output["homepage_url"].apply(base_domain)
output["resource_list_url"] = ""
output["resource_list_notes"] = ""

# Prioritize Step 5 work by biomedical relevance, not alphabetically (see the cell above) --
# institution_name is only a tiebreaker for institutions with the same pct_biomedical.
output = output.sort_values(["pct_biomedical", "institution_name"], ascending=[False, True]).reset_index(drop=True)

# Dated filename (not a fixed name) -- each re-run of this pipeline (e.g. after a reference-list
# fix, or just to pick up newly-added institutions) produces its own dated snapshot rather than
# silently overwriting the previous one, so two runs can be diffed to see exactly what changed
# (see the "Comparing runs" section in ../SKILL.md).
OUT_PATH = f"../../output/{date.today().isoformat()}_accredited_nonprofit_secular_institutions.xlsx"
output.to_excel(OUT_PATH, index=False)
OUT_PATH, output.shape

('../../output/2026-07-31_accredited_nonprofit_secular_institutions.xlsx',
 (2399, 11))

## De-duplicate by homepage domain before Step 5

Multiple rows can share the same `base_domain` — typically branch/satellite campuses or
multi-location systems (e.g. a correctional-facility campus of the same college, or two campuses of
the same university system) that are served by one shared library site. Step 5 should search **once
per unique `base_domain`**, not once per institution row, and then apply the result to every row
sharing that domain (see `../SKILL.md`).

In [10]:
n_institutions = len(output)
n_domains = output["base_domain"].nunique()
dup_rows = output[output.duplicated("base_domain", keep=False)].sort_values("base_domain")

print(f"{n_institutions} institutions -> {n_domains} unique base domains "
      f"({n_institutions - n_domains} fewer Step 5 searches needed)")
dup_rows[["institution_name", "base_domain", "unitid"]]

2399 institutions -> 2109 unique base domains (290 fewer Step 5 searches needed)


,institution_name,base_domain,unitid
1932,Northwest Vista College,alamo.edu,420398
1318,St Philip's College,alamo.edu,227854
2233,Northeast Lakeview College,alamo.edu,488730
1400,San Antonio College,alamo.edu,227924
1900,Palo Alto College,alamo.edu,246354
...,...,...,...
2369,University of the Virgin Islands-Albert A. Sheen,uvi.edu,24366501
1621,Victor Valley College,vvc.edu,125091
2376,Victor Valley Community College - Aviation Tec...,vvc.edu,12509101
2339,The Wright Institute,wi.edu,126012


## Next: Step 5

For each **unique `base_domain`** in the exported spreadsheet (not each row), find that
institution's library resources/databases page (or a biomedical/health-sciences subject guide) and
fill in `resource_list_url` (and `resource_list_notes` for anything ambiguous or not found) for
every row sharing that domain. See `../SKILL.md` for the search procedure and how
`scripts/update_resource_entry.py --domain ...` applies one result to all matching rows at once —
this part is done by an agent with web search/browse tools, not by this notebook.